# Dataset inspection

This notebook inspects the retail transaction data for:
- data types
- missing values
- unusual quantities and prices
- cancelled transactions

In [2]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

try:
    import openpyxl  # noqa: F401
except ImportError:
    print("Installing openpyxl for Excel support...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    import openpyxl  # noqa: F401

repo_root = Path.cwd()
if not (repo_root / "data" / "raw" / "Online Retail.xlsx").exists():
    repo_root = repo_root.parent

DATA_PATH = repo_root / "data" / "raw" / "Online Retail.xlsx"
print(f"Loading dataset from: {DATA_PATH}")

excel_file = pd.ExcelFile(DATA_PATH)
print("Available sheets:", excel_file.sheet_names)

df = pd.read_excel(DATA_PATH, sheet_name=excel_file.sheet_names[0])
print(f"Shape: {df.shape}")
df.head()

Loading dataset from: /home/vinuri/Documents/InsightRetail/data/raw/Online Retail.xlsx
Available sheets: ['Online Retail']
Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# 1) Data types
print("Data types:\n")
print(df.dtypes)

print("\nPreview of columns:")
print(list(df.columns))

Data types:

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Preview of columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [4]:
# 2) Missing values
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

print("Missing values summary:\n")
print(missing_summary)

Missing values summary:

             missing_count  missing_percent
CustomerID          135080            24.93
Description           1454             0.27
StockCode                0             0.00
InvoiceNo                0             0.00
Quantity                 0             0.00
InvoiceDate              0             0.00
UnitPrice                0             0.00
Country                  0             0.00


In [5]:
# 3) Unusual quantities and prices
quantity_series = pd.to_numeric(df["Quantity"], errors="coerce")
price_series = pd.to_numeric(df["UnitPrice"], errors="coerce")

print("Quantity summary:\n")
print(quantity_series.describe())

print("\nUnit price summary:\n")
print(price_series.describe())

# IQR-based outlier detection
q1_qty, q3_qty = quantity_series.quantile([0.25, 0.75])
iqr_qty = q3_qty - q1_qty
lower_qty = q1_qty - 1.5 * iqr_qty
upper_qty = q3_qty + 1.5 * iqr_qty

q1_price, q3_price = price_series.quantile([0.25, 0.75])
iqr_price = q3_price - q1_price
lower_price = q1_price - 1.5 * iqr_price
upper_price = q3_price + 1.5 * iqr_price

unusual_quantity = df[(quantity_series < lower_qty) | (quantity_series > upper_qty)]
unusual_price = df[(price_series < lower_price) | (price_series > upper_price)]

print(f"\nUnusual quantity rows: {len(unusual_quantity)}")
print(unusual_quantity[["InvoiceNo", "StockCode", "Description", "Quantity"]].head(10))

print(f"\nUnusual price rows: {len(unusual_price)}")
print(unusual_price[["InvoiceNo", "StockCode", "Description", "UnitPrice"]].head(10))

Quantity summary:

count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

Unit price summary:

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

Unusual quantity rows: 58619
   InvoiceNo StockCode                          Description  Quantity
9     536367     84879        ASSORTED COLOUR BIRD ORNAMENT        32
26    536370     22728            ALARM CLOCK BAKELIKE PINK        24
27    536370     22727            ALARM CLOCK BAKELIKE RED         24
30    536370     21883                     STARS GIFT TAPE         24
31    536370     10002          INFLATABLE POLITICAL GLOBE         48
32    536370     21791   VINTAGE HEADS AND TAILS CARD GAME         24
34    53

In [6]:
# 4) Cancelled transactions
# In retail data, cancelled invoices are often prefixed with 'C'
df["InvoiceNo_str"] = df["InvoiceNo"].astype(str)

cancelled_transactions = df[df["InvoiceNo_str"].str.startswith("C", na=False)].copy()

print(f"Cancelled transaction count: {len(cancelled_transactions)}")
print(cancelled_transactions[["InvoiceNo", "Quantity", "UnitPrice", "CustomerID"]].head(10))

Cancelled transaction count: 9288
    InvoiceNo  Quantity  UnitPrice  CustomerID
141   C536379        -1      27.50     14527.0
154   C536383        -1       4.65     15311.0
235   C536391       -12       1.65     17548.0
236   C536391       -24       0.29     17548.0
237   C536391       -24       0.29     17548.0
238   C536391       -24       0.29     17548.0
239   C536391       -12       3.45     17548.0
240   C536391       -12       1.65     17548.0
241   C536391       -24       1.65     17548.0
939   C536506        -6       4.25     17897.0
